# 画像 + 動きの指示文から、動画を作る(CogVideoX-5B-I2V・Google Colab)
SVD は「揺れるだけ」で、動きの指示ができません。こちらは**英語の指示文で動きを指定できます**(例: 手を振る、笑う、髪が風になびく)。
**未検証**: この環境にはGPUが無いので、動作確認はできていません。メモリ不足・時間切れが起きるかもしれません。起きたら、画面のエラーをそのまま送ってください。

## 使い方
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. セルを上から順に ▶ で実行(モデルの取得に数分、生成に**数十分**かかる見込み)。
3. 画像のアップロードで、キャラの画像を選ぶ。
4. 最後のセルで、動画(mp4・約6秒・720x480の横長)がダウンロードできます。

## 動きの指示文(英語)の書き方
- 人物と動きを、具体的に・短く。例: `The girl smiles and waves her right hand, her long black hair gently blowing in the wind.`
- 腕の交差や、大きな動きは失敗しやすいです。まず小さな動き(手を振る・うなずく・髪がなびく)から。
- 口パクは、このAIでは正確にできません(口がもごもご動く程度)。

In [ ]:
!pip -q install diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
# T4 は bfloat16 が得意でないので、使える型を自動で選ぶ
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe = CogVideoXImageToVideoPipeline.from_pretrained("THUDM/CogVideoX-5b-I2V", torch_dtype=dtype)
pipe.enable_sequential_cpu_offload()   # GPUメモリを節約(遅くなる)
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print("型:", dtype)

In [ ]:
from google.colab import files
from PIL import Image, ImageFilter
up = files.upload()
img0 = Image.open(list(up.keys())[0]).convert("RGB")
W, H = 720, 480
# 全身の縦長画像は、切らずに、横長の中央に置く(左右は同じ画像をぼかして埋める)
bg = img0.resize((W, round(img0.height * W / img0.width)), Image.LANCZOS)
bg = bg.crop((0, (bg.height - H) // 2, W, (bg.height - H) // 2 + H)).filter(ImageFilter.GaussianBlur(25))
fg = img0.resize((round(img0.width * H / img0.height), H), Image.LANCZOS)
img = bg.copy()
img.paste(fg, ((W - fg.width) // 2, 0))
img

In [ ]:
print('生成を始めます。数十分かかります(進み具合のバーが出ます)。このセルが ▶ の回転中なら動いています。')
from diffusers.utils import export_to_video
# 動きの指示文(英語)。歩く動き。服や色は、画像に合わせて書き換える
prompt = ("A full-body anime girl walks forward confidently toward the camera, arms swinging naturally, "
          "her hair and skirt flowing, smooth walking motion, anime style, high quality")
frames = pipe(image=img, prompt=prompt, num_videos_per_prompt=1, num_inference_steps=30, num_frames=49,
              guidance_scale=6, generator=torch.Generator().manual_seed(1)).frames[0]
export_to_video(frames, "clip_cog.mp4", fps=8)

### ここで、できた動画が下に表示されます(▶ で再生)。表示されないときは、上のセルにエラーが出ていないか確認してください。

In [ ]:
import os
print("動画ファイル:", os.path.exists("clip_cog.mp4"), os.path.getsize("clip_cog.mp4") if os.path.exists("clip_cog.mp4") else 0, "バイト")
from IPython.display import Video, display
display(Video("clip_cog.mp4", embed=True, width=480))

In [ ]:
# なめらかにして(8fps → 24fps)ダウンロード
!ffmpeg -y -loglevel error -i clip_cog.mp4 -vf "minterpolate=fps=24:mi_mode=mci" -pix_fmt yuv420p clip_cog_smooth.mp4
from google.colab import files
files.download("clip_cog_smooth.mp4")

`clip_cog_smooth.mp4` を、チャットにアップロードしてください。声・吹き出し・BGMを重ねて仕上げます。